# SEC Filing Anomaly Detection — Deliverable 2

**Objective:** identify unusual company-quarter financial patterns using statistical Z-scores and Isolation Forest, compare the results with the reference/control groups, and use the annual Beneish M-Score as a separate screening baseline.

**Interpretation rule:** an anomaly is a review candidate, not evidence that fraud occurred. Legitimate operating events, XBRL tagging changes and data quality problems can also produce outliers.

This notebook builds on Deliverable 1's `quarterly_features.csv` and `annual_features.csv`. It does not re-download SEC data by default.

In [ ]:
from pathlib import Path
import sys
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display

# Works when launched from the repository root or the notebook's parent directory.
CWD = Path.cwd()
REPO_ROOT = CWD if (CWD / "src" / "anomaly_pipeline.py").exists() else CWD.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
DATA_DIR = REPO_ROOT / "data" / "processed"
print("Repository:", REPO_ROOT)
print("Data directory:", DATA_DIR)
if not (DATA_DIR / "quarterly_features.csv").exists():
    raise FileNotFoundError("Expected data/processed/quarterly_features.csv. Run Deliverable 1 first.")

## 1. Inspect the input data

We first inspect the dataset dimensions and key columns. Feature generation was done in Deliverable 1, so this stage consumes the existing secondary dataset.

In [ ]:
quarterly = pd.read_csv(DATA_DIR / "quarterly_features.csv", low_memory=False)
annual_path = DATA_DIR / "annual_features.csv"
annual = pd.read_csv(annual_path, low_memory=False) if annual_path.exists() else None
print(f"Quarterly data: {quarterly.shape[0]:,} rows × {quarterly.shape[1]} columns")
print(f"Annual data: {annual.shape[0]:,} rows × {annual.shape[1]} columns" if annual is not None else "Annual data unavailable")
display(quarterly[[c for c in ["ticker", "company", "group", "fiscal_year", "fiscal_quarter", "period_end", "revenue", "revenue_yoy_growth", "total_expense_to_revenue"] if c in quarterly.columns]].head())

## 2. Review data-quality flags

The current extraction notebook reported some fiscal-year quarter-sum mismatches. For the first modelling pass, rows with serious `dq_*` flags are excluded instead of spending time reconciling every historical discrepancy. We also exclude rows where the revenue XBRL tag changed, since a tag change may distort growth metrics.

The original CSV is not modified; the filtered rows and counts remain inspectable.

In [ ]:
from src.anomaly_pipeline import (
    prepare_observations, select_features, robust_zscore_results,
    isolation_forest_results, attach_beneish_baseline, summarize_results,
    build_beneish_comparison, run_detection,
)

clean, filter_stats = prepare_observations(
    quarterly,
    exclude_quality_flags=True,
    exclude_tag_switches=True,
)
print(json.dumps(filter_stats, indent=2))
quality_cols = [c for c in [
    "dq_negative_revenue", "dq_missing_revenue", "dq_period_length_unusual",
    "dq_quarter_looks_annual", "dq_quarter_sum_mismatch", "revenue_tag_changed"
] if c in quarterly.columns]
if quality_cols:
    quality_summary = pd.DataFrame({
        "flag": quality_cols,
        "flagged_rows": [int(quarterly[c].astype(str).str.lower().isin(["true", "1", "yes"]).sum()) for c in quality_cols],
        "percent_of_input": [float(quarterly[c].astype(str).str.lower().isin(["true", "1", "yes"]).mean() * 100) for c in quality_cols],
    })
    display(quality_summary)

## 3. Choose the model features

The model uses growth rates, margins, expense ratios, working-capital signals, accruals/cash conversion and trailing Z-score features. Company identifiers, the reference/control label and restatement outcome fields are not model inputs. `group` is retained only for post-fit descriptive comparisons.

In [ ]:
feature_columns = select_features(clean)
print(f"Selected {len(feature_columns)} model features:")
for start in range(0, len(feature_columns), 6):
    print("  " + ", ".join(feature_columns[start:start+6]))

## 4. Statistical detection and Isolation Forest

- **Statistical score:** maximum absolute robust trailing Z-score across available precomputed metrics (ordinary trailing Z-scores are a fallback). A default threshold of 3.5 marks especially unusual observations.
- **Isolation Forest:** 300 trees, fixed random seed for reproducibility and a default `contamination=0.05`. This flags approximately the most unusual 5% of retained observations according to the model's decision threshold; 5% is a configuration choice, not the prevalence of fraud.
- Missing model features are median-imputed. Features with excessive missingness are excluded from the candidate set.

In [ ]:
Z_THRESHOLD = 3.5
CONTAMINATION = 0.05

scored = robust_zscore_results(clean, threshold=Z_THRESHOLD)
scored, isolation_forest_model = isolation_forest_results(
    scored, feature_columns, contamination=CONTAMINATION, random_state=42
)
# Annual Beneish information is attached after fitting for evaluation only.
scored = attach_beneish_baseline(scored, annual)
model_comparison, company_ranking = summarize_results(scored)
beneish_comparison = build_beneish_comparison(scored)

print(f"Observations scored: {len(scored):,}")
print(f"Isolation Forest flags: {int(scored['if_anomaly'].sum()):,}")
print(f"Statistical flags: {int(scored['statistical_anomaly'].sum()):,}")
display(model_comparison)

## 5. Review the highest-ranked observations

The table is sorted by Isolation Forest score, with larger values indicating more unusual patterns. Review the contributing financial metrics and filing history before interpreting any observation.

In [ ]:
review_columns = [c for c in [
    "ticker", "company", "group", "period_end", "fiscal_year", "fiscal_quarter",
    "revenue", "revenue_yoy_growth", "total_expense_to_revenue", "gross_margin",
    "dso", "accruals_to_assets", "ar_growth_minus_revenue_growth",
    "statistical_anomaly_score", "statistical_anomaly", "if_anomaly_score", "if_anomaly",
    "beneish_m_score", "beneish_flag"
] if c in scored.columns]
top_observations = scored.sort_values("if_anomaly_score", ascending=False).head(25)
display(top_observations[review_columns])

## 6. Compare methods and reference/control groups

The group column represents company-level weak labels, not quarter-level ground truth. Rates in this table are descriptive. They must not be presented as accuracy, precision or recall.

In [ ]:
display(model_comparison)
if "group" in scored.columns:
    group_comparison = scored.groupby("group", dropna=False).agg(
        observations=("ticker", "size"),
        isolation_forest_flag_rate=("if_anomaly", "mean"),
        statistical_flag_rate=("statistical_anomaly", "mean"),
        median_if_score=("if_anomaly_score", "median"),
    ).reset_index()
    group_comparison[["isolation_forest_flag_rate", "statistical_flag_rate"]] *= 100
    group_comparison = group_comparison.rename(columns={
        "isolation_forest_flag_rate": "IF_flag_rate_pct",
        "statistical_flag_rate": "statistical_flag_rate_pct",
    })
    display(group_comparison)

display(company_ranking.head(20))

## 7. Annual Beneish baseline

Beneish is calculated from annual data and therefore is not a like-for-like quarter-level label. The comparison aggregates quarterly alerts to company-year and checks whether at least one quarter was flagged in a year that also crosses the conventional Beneish threshold. This is a screening comparison, not a validation score.

In [ ]:
if not beneish_comparison.empty:
    display(beneish_comparison.sort_values("if_max_score", ascending=False).head(25))
    available = beneish_comparison.dropna(subset=["beneish_flag"])
    if not available.empty:
        print("Company-years with Beneish M-Score available:", len(available))
        print("Company-years flagged by Beneish:", int(available["beneish_flag"].astype(bool).sum()))
        print("Company-years with at least one IF-flagged quarter:", int(available["if_any_quarter_flagged"].astype(bool).sum()))
else:
    print("Beneish comparison unavailable: expected annual columns were not found.")

## 8. Visualize the results

These plots are meant for triage: they reveal concentration and candidate cases for manual review. They do not establish intent or fraud.

In [ ]:
# Plot 1: the distribution of Isolation Forest scores
plt.figure(figsize=(8, 4))
plt.hist(scored["if_anomaly_score"].dropna(), bins=35)
plt.xlabel("Isolation Forest anomaly score (higher = more unusual)")
plt.ylabel("Company-quarter count")
plt.title("Distribution of Isolation Forest scores")
plt.tight_layout()
plt.show()

# Plot 2: top 15 company-quarter observations
plot_df = scored.nlargest(15, "if_anomaly_score").copy()
plot_df["label"] = plot_df["ticker"].astype(str) + " " + plot_df["period_end"].astype(str).str[:10]
plot_df = plot_df.sort_values("if_anomaly_score")
plt.figure(figsize=(10, 6))
plt.barh(plot_df["label"], plot_df["if_anomaly_score"])
plt.xlabel("Isolation Forest anomaly score")
plt.ylabel("Company and quarter end")
plt.title("Top 15 company-quarter review candidates")
plt.tight_layout()
plt.show()

## 9. Export results

The same processing functions are also called by `run_pipeline.py`, so notebook and command-line results share the same implementation.

In [ ]:
results = run_detection(
    DATA_DIR,
    output_dir=DATA_DIR,
    contamination=CONTAMINATION,
    z_threshold=Z_THRESHOLD,
)
print(json.dumps(results["summary"], indent=2))
print("Generated files:")
for name in [
    "scored_quarterly.csv", "anomaly_top100.csv", "model_comparison.csv",
    "company_anomaly_ranking.csv", "beneish_comparison.csv", "run_summary.json"
]:
    path = DATA_DIR / name
    print(f"{'OK' if path.exists() else 'MISSING'}  {path}")

## 10. Conclusion and limitations

The outputs create a shortlist of unusual financial observations for investigation. Before making a case-level claim, inspect the original filings, filing dates/accession numbers, XBRL-tag changes, business events and restatement history. A stronger evaluation would require event-dated quarter-level labels and a prospective or point-in-time test. This notebook's group labels are weak labels only.